In [1]:
import os
import ot
import gc
import k3d
import torch
import trimesh
import warnings
from tqdm import *
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import scipy.sparse as sp
import matplotlib.cm as cm
from anndata import AnnData
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from numpy.random import RandomState
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
from sklearn.metrics import jaccard_score
from scipy.stats import fisher_exact, norm
from sklearn.neighbors import NearestNeighbors
from typing import Literal, Optional, Tuple, Union
from matplotlib.colors import ListedColormap, rgb2hex
from sklearn.metrics.pairwise import euclidean_distances
from matplotlib.font_manager import fontManager, FontProperties
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

import marsilea as ma
import re
from matplotlib_scalebar.scalebar import ScaleBar

plt.rcParams['pdf.fonttype'] = 42
import matplotlib.colors as mcolors
from matplotlib.font_manager import fontManager, FontProperties
import os
fontManager.addfont('/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial.ttf')
font = FontProperties(fname='/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name


tick_font = FontProperties(fname='/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial-ItalicMT.otf', style = 'italic')

In [2]:
adata_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/merfish_mouseBrain_concat_embeddings.h5ad"
csv_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/example/05_merfish_mouseBrain/cluster_to_cluster_annotation_membership.csv"
json_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/author_colormaps/author_colormap.json"


adata = sc.read_h5ad(adata_path)
df = pd.read_csv(csv_path)

def strip_author_id(x):
    return re.sub(r"^\s*\d+\s+", "", str(x)).strip()

class_df = df[df["cluster_annotation_term_set_name"] == "class"].copy()
class_df["cluster_alias"] = class_df["cluster_alias"].astype(str)
class_df["author_class"] = class_df["cluster_annotation_term_name"].map(strip_author_id)

cluster_to_class = (
    class_df
    .drop_duplicates("cluster_alias")
    .set_index("cluster_alias")["author_class"]
    .to_dict()
)

adata.obs["author_class"] = (
    adata.obs["cluster_id_transfer"]
    .astype(str)
    .map(cluster_to_class)
    .astype("category")
)

In [3]:
import json

with open(json_path, "r", encoding="utf-8") as f:
    author_cmap = json.load(f)

print(author_cmap.keys())

class_palette = author_cmap["author_term_set_palettes"]["class"]
cluster_palette = author_cmap["obs_key_palettes"]["cluster_id_transfer"]
subclass_palette = author_cmap["obs_key_palettes"]["subclass_transfer"]

donor_id_colormap = {
    'C57BL6J-1': '#73BBF4', 
    'C57BL6J-2': '#284D76', 
    'C57BL6J-3': '#DF95D5', 
    'C57BL6J-4': '#791E25',
}

dict_keys(['source_membership', 'source_h5ad', 'obs_key_palettes', 'author_term_set_palettes'])


In [4]:
len(subclass_palette)

338

In [5]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb, to_hex


def blend_with_white(hex_color, amount=0.0):
    """
    amount=0: 原色
    amount=1: 白色
    """
    rgb = np.array(to_rgb(hex_color))
    white = np.array([1, 1, 1])
    new_rgb = rgb * (1 - amount) + white * amount
    return to_hex(new_rgb)


def build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="author_class",
    class_palette=None,
    min_lighten=0.0,
    max_lighten=0.55,
):
    obs = adata.obs[[leiden_key, author_key]].dropna().copy()

    obs[leiden_key] = obs[leiden_key].astype(str)
    obs[author_key] = obs[author_key].astype(str)

    # Leiden -> author_class composition
    confusion = pd.crosstab(
        obs[leiden_key],
        obs[author_key],
        normalize="index"
    )

    # 每个 Leiden 最相关的 author_class
    best_class = confusion.idxmax(axis=1)
    best_score = confusion.max(axis=1)

    mapping_df = pd.DataFrame({
        "cell_leiden": confusion.index,
        "matched_author_class": best_class.values,
        "matched_fraction": best_score.values,
    })

    leiden_palette = {}

    for author_class, sub_df in mapping_df.groupby("matched_author_class"):
        if class_palette is None or author_class not in class_palette:
            base_color = "#808080"
        else:
            base_color = class_palette[author_class]

        # 同一个 author_class 下，匹配度最高的 Leiden 最接近原色
        sub_df = sub_df.sort_values("matched_fraction", ascending=False)

        n = len(sub_df)

        for rank, (_, row) in enumerate(sub_df.iterrows()):
            leiden = row["cell_leiden"]

            if n == 1:
                lighten = min_lighten
            else:
                lighten = min_lighten + (max_lighten - min_lighten) * rank / (n - 1)

            leiden_palette[leiden] = blend_with_white(base_color, amount=lighten)

    return leiden_palette, mapping_df, confusion

In [6]:
import re
import numpy as np
import pandas as pd

try:
    from scipy.stats import fisher_exact
except Exception:
    fisher_exact = None


def auto_find_enriched_groups(
    adata,
    group_key,
    target_key,
    target_regex=None,
    target_values=None,
    section_slices=None,
    section_key="brain_section_label",
    min_group_cells=30,
    min_target_cells=10,
    min_enrichment=2.0,
    min_frac_group_in_target=0.20,
    max_groups=30,
    use_fdr=True,
    verbose=True,
):
    """
    一行自动寻找某个目标区域富集的 group。

    例子：
    groups, table = auto_find_enriched_groups(
        adata,
        group_key="niche_leiden",
        target_key="cluster_annotation",
        target_regex="cerebell|cerebellum|CB|Cb",
        section_slices=["C57BL6J-2.060", "C57BL6J-3.016", "C57BL6J-3.004"],
    )
    """

    def _bh_fdr(pvals):
        pvals = np.asarray(pvals, dtype=float)
        order = np.argsort(pvals)
        out = np.empty_like(pvals)
        n = len(pvals)
        prev = 1.0
        for i in range(n - 1, -1, -1):
            rank = i + 1
            val = pvals[order[i]] * n / rank
            prev = min(prev, val)
            out[order[i]] = prev
        return np.clip(out, 0, 1)

    if group_key not in adata.obs:
        raise KeyError(f"{group_key} not found in adata.obs")
    if target_key not in adata.obs:
        raise KeyError(f"{target_key} not found in adata.obs")
    if section_slices is not None and section_key not in adata.obs:
        raise KeyError(f"{section_key} not found in adata.obs")

    group = adata.obs[group_key].astype("string").fillna("NA").astype(str).to_numpy()
    target_label = adata.obs[target_key].astype("string").fillna("NA").astype(str)

    target_mask = np.zeros(adata.n_obs, dtype=bool)

    if target_values is not None:
        target_values = [str(x) for x in target_values]
        target_mask |= target_label.isin(target_values).to_numpy()

    if target_regex is not None:
        target_mask |= target_label.str.contains(
            target_regex,
            case=False,
            regex=True,
            na=False,
        ).to_numpy()

    if target_values is None and target_regex is None:
        raise ValueError("Please provide target_regex or target_values")

    if section_slices is not None:
        section_slices = [str(x) for x in section_slices]
        section_mask = adata.obs[section_key].astype(str).isin(section_slices).to_numpy()
    else:
        section_mask = np.ones(adata.n_obs, dtype=bool)

    background_mask = section_mask
    target_mask = target_mask & section_mask

    n_target_total = int(target_mask.sum())
    n_bg_total = int((background_mask & ~target_mask).sum())

    if n_target_total == 0:
        examples = (
            target_label.value_counts()
            .head(30)
            .rename_axis(target_key)
            .reset_index(name="n_cells")
        )
        raise ValueError(
            f"target_mask selected 0 cells. Check target_regex/target_values. "
            f"Top {target_key} values:\n{examples}"
        )

    rows = []
    for g in sorted(pd.unique(group[background_mask]).astype(str)):
        in_group = background_mask & (group == g)

        n_group = int(in_group.sum())
        n_group_target = int((in_group & target_mask).sum())
        n_group_bg = int((in_group & ~target_mask).sum())

        if n_group < min_group_cells or n_group_target < min_target_cells:
            continue

        frac_group_in_target = n_group_target / max(n_group, 1)
        frac_target_covered = n_group_target / max(n_target_total, 1)
        frac_group_in_background = n_group_bg / max(n_bg_total, 1)
        enrichment = (frac_target_covered + 1e-12) / (frac_group_in_background + 1e-12)

        if fisher_exact is not None:
            table = [
                [n_group_target, n_target_total - n_group_target],
                [n_group_bg, n_bg_total - n_group_bg],
            ]
            _, pval = fisher_exact(table, alternative="greater")
        else:
            pval = np.nan

        rows.append({
            "group": str(g),
            "n_group": n_group,
            "n_group_target": n_group_target,
            "n_group_background": n_group_bg,
            "frac_group_in_target": frac_group_in_target,
            "frac_target_covered": frac_target_covered,
            "enrichment": enrichment,
            "pval": pval,
        })

    table = pd.DataFrame(rows)

    if table.empty:
        if verbose:
            print("No group passed min_group_cells/min_target_cells.")
        return [], table

    if table["pval"].notna().any():
        table["fdr"] = _bh_fdr(table["pval"].fillna(1).to_numpy())
    else:
        table["fdr"] = np.nan

    table = table.sort_values(
        ["enrichment", "frac_group_in_target", "n_group_target"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    picked = table[
        (table["enrichment"] >= min_enrichment)
        & (table["frac_group_in_target"] >= min_frac_group_in_target)
        & (table["n_group_target"] >= min_target_cells)
    ].copy()

    if use_fdr and picked["fdr"].notna().any():
        picked = picked[picked["fdr"] <= 0.05]

    groups = picked.head(max_groups)["group"].astype(str).tolist()

    if verbose:
        print(f"target cells: {n_target_total:,}")
        print(f"background cells: {int(background_mask.sum()):,}")
        print(f"selected {len(groups)} groups from {group_key}:")
        print(groups)

    return groups, table

In [7]:
adata = adata[
    adata.obs["major_brain_region"].notna()
    & (adata.obs["major_brain_region"] != "n/a")
].copy()

In [8]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb, to_hex
import colorsys


def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))


def tweak_color(
    hex_color,
    lighten=0.0,
    hue_shift=0.0,
    sat_scale=1.0,
):
    """
    在保留原始色系的基础上，调整：
    lighten: 向白色混合，0 原色，1 白色
    hue_shift: 色相偏移，建议小范围，如 -0.06 到 0.06
    sat_scale: 饱和度缩放，>1 更鲜艳，<1 更灰
    """
    r, g, b = to_rgb(hex_color)

    h, l, s = colorsys.rgb_to_hls(r, g, b)

    h = (h + hue_shift) % 1.0
    s = clamp(s * sat_scale)

    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)

    rgb = np.array([r2, g2, b2])
    white = np.array([1, 1, 1])

    new_rgb = rgb * (1 - lighten) + white * lighten
    return to_hex(new_rgb)


def symmetric_offsets(n, max_shift):
    """
    生成类似：
    0, +1, -1, +2, -2 ...
    这样最重要的 cluster 最接近原色，后面的逐渐偏移。
    """
    if n == 1 or max_shift == 0:
        return [0.0] * n

    offsets = [0.0]
    step = max_shift / max(1, np.ceil((n - 1) / 2))

    k = 1
    while len(offsets) < n:
        offsets.append(k * step)
        if len(offsets) < n:
            offsets.append(-k * step)
        k += 1

    return offsets[:n]


def build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="author_class",
    class_palette=None,
    min_lighten=0.0,
    max_lighten=0.45,
    max_hue_shift=0.07,
    min_sat_scale=0.75,
    max_sat_scale=1.15,
):
    obs = adata.obs[[leiden_key, author_key]].dropna().copy()

    obs[leiden_key] = obs[leiden_key].astype(str)
    obs[author_key] = obs[author_key].astype(str)

    confusion = pd.crosstab(
        obs[leiden_key],
        obs[author_key],
        normalize="index"
    )

    best_class = confusion.idxmax(axis=1)
    best_score = confusion.max(axis=1)

    mapping_df = pd.DataFrame({
        "cell_leiden": confusion.index,
        "matched_author_class": best_class.values,
        "matched_fraction": best_score.values,
    })

    leiden_palette = {}

    for author_class, sub_df in mapping_df.groupby("matched_author_class"):
        if class_palette is None or author_class not in class_palette:
            base_color = "#808080"
        else:
            base_color = class_palette[author_class]

        # 匹配度最高的 Leiden 最接近 author_class 原色
        sub_df = sub_df.sort_values("matched_fraction", ascending=False)

        n = len(sub_df)
        hue_offsets = symmetric_offsets(n, max_hue_shift)

        for rank, (_, row) in enumerate(sub_df.iterrows()):
            leiden = row["cell_leiden"]

            if n == 1:
                lighten = min_lighten
                sat_scale = 1.0
            else:
                t = rank / (n - 1)

                # 亮度逐渐增加
                lighten = min_lighten + (max_lighten - min_lighten) * t

                # 饱和度做轻微变化：前面的更鲜明，后面的略灰一些
                sat_scale = max_sat_scale + (min_sat_scale - max_sat_scale) * t

            leiden_palette[leiden] = tweak_color(
                base_color,
                lighten=lighten,
                hue_shift=hue_offsets[rank],
                sat_scale=sat_scale,
            )

    return leiden_palette, mapping_df, confusion

In [9]:
cell_leiden_palette, leiden_author_map, confusion = build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="subclass_transfer",
    class_palette=subclass_palette,
    min_lighten=0.20,
    max_lighten=0.50,
    max_hue_shift=2.00,
    min_sat_scale=0.50,
    max_sat_scale=100.00
)

In [10]:
leiden_author_map.to_csv('/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/leiden_author_map.csv')

In [11]:
section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None
                    # groups = ob_cells
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/all/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [12]:
section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None
                    # groups = ob_cells
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/all/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [13]:
section_slices = ['C57BL6J-2.060', 'C57BL6J-3.016', 'C57BL6J-3.004']

cb_author_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="subclass_transfer",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Cerebellum",
    section_slices=section_slices,
)

cb_leiden_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="cell_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Cerebellum",
    section_slices=section_slices,
)

target cells: 47,168
background cells: 244,267
selected 14 groups from subclass_transfer:
['CBX Purkinje Gaba', 'Bergmann NN', 'CBX MLI Megf11 Gaba', 'CBX MLI Cdh22 Gaba', 'CB PLI Gly-Gaba', 'CBX Golgi Gly-Gaba', 'CB Granule Glut', 'Astro-CB NN', 'CBN Dmbx1 Gaba', 'DCO UBC Glut', 'VCO Mafa Meis2 Glut', 'SPVI-SPVC Sall3 Lhx1 Gly-Gaba', 'Astroependymal NN', 'CBN Neurod2 Pvalb Glut']
target cells: 47,168
background cells: 244,267
selected 30 groups from cell_leiden:
['341', '338', '278', '305', '216', '129', '326', '306', '330', '334', '96', '255', '287', '336', '310', '267', '317', '50', '333', '316', '322', '252', '190', '266', '213', '210', '161', '42', '14', '5']


In [14]:
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_author_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/cb_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [15]:
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_leiden_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/cb_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [16]:
section_slices = ['C57BL6J-1.022', 'C57BL6J-1.030', 'C57BL6J-3.006']

ob_author_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="subclass_transfer",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Olfactory",
    section_slices=section_slices,
)


ob_leiden_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="cell_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Olfactory",
    section_slices=section_slices,
)


for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_author_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ob_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_leiden_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ob_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 45,535
background cells: 157,306
selected 19 groups from subclass_transfer:
['OB Trdn Gaba', 'OB-mi Frmd7 Gaba', 'OB Eomes Ms4a15 Glut', 'LSX Sall3 Pax6 Gaba', 'OB-in Frmd7 Gaba', 'OB Meis2 Thsd7b Gaba', 'Astro-OLF NN', 'OB Dopa-Gaba', 'IT AON-TT-DP Glut', 'OB-out Frmd7 Gaba', 'L2/3 IT PIR-ENTl Glut', 'IA Mgp Gaba', 'PVHd-SBPV Six3 Prox1 Gaba', 'OB-STR-CTX Inh IMN', 'NDB-SI-ant Prdm12 Gaba', 'IT EP-CLA Glut', 'MEA-COA-BMA Ccdc42 Glut', 'PRC-PAG Tcf7l2 Irx2 Glut', 'STR-PAL Chst9 Gaba']
target cells: 45,535
background cells: 157,306
selected 30 groups from cell_leiden:
['335', '218', '79', '236', '265', '193', '187', '324', '162', '240', '66', '164', '183', '165', '248', '250', '300', '145', '81', '222', '29', '276', '40', '86', '185', '313', '234', '118', '123', '184']


In [17]:
section_slices = ['C57BL6J-1.080', 'C57BL6J-3.007', 'C57BL6J-3.015']

ob_author_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="subclass_transfer",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Isocortex",
    section_slices=section_slices,
)

ob_leiden_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="cell_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Isocortex",
    section_slices=section_slices,
)


for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_author_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ctx_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

section_slices = ['C57BL6J-1.080', 'C57BL6J-3.007', 'C57BL6J-3.015']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_leiden_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/ctx_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 54,581
background cells: 274,120
selected 17 groups from subclass_transfer:
['L5 IT CTX Glut', 'L4/5 IT CTX Glut', 'L5 ET CTX Glut', 'L2/3 IT CTX Glut', 'L5 NP CTX Glut', 'L6 IT CTX Glut', 'L2/3 IT RSP Glut', 'L6 CT CTX Glut', 'CLA-EPd-CTX Car3 Glut', 'L4 RSP-ACA Glut', 'L5/6 IT TPE-ENT Glut', 'Pvalb Gaba', 'Lamp5 Gaba', 'Vip Gaba', 'Sst Gaba', 'ABC NN', 'Sncg Gaba']
target cells: 54,581
background cells: 274,120
selected 30 groups from cell_leiden:
['171', '253', '133', '191', '312', '241', '172', '214', '247', '243', '199', '181', '232', '127', '245', '192', '284', '196', '295', '116', '154', '170', '112', '229', '318', '56', '37', '130', '319', '177']


In [18]:
section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']

ob_author_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="subclass_transfer",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Hippocampus",
    section_slices=section_slices,
)

ob_leiden_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="cell_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Hippocampus",
    section_slices=section_slices,
)


for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_author_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/hip_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_leiden_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/hip_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 19,440
background cells: 210,267
selected 20 groups from subclass_transfer:
['CA3 Glut', 'CA1-ProS Glut', 'DG Glut', 'CA2-FC-IG Glut', 'SUB-ProS Glut', 'NP SUB Glut', 'DG-PIR Ex IMN', 'L2 IT PPP-APr Glut', 'HPF CR Glut', 'RHP-COA Ndnf Gaba', 'L2/3 IT PPP Glut', 'Lamp5 Lhx6 Gaba', 'L2/3 IT ENT Glut', 'L6b/CT ENT Glut', 'Pvalb chandelier Gaba', 'Sncg Gaba', 'ENTmv-PA-COAp Glut', 'Astro-TE NN', 'CT SUB Glut', 'Sst Gaba']
target cells: 19,440
background cells: 210,267
selected 30 groups from cell_leiden:
['311', '286', '168', '137', '139', '151', '242', '259', '296', '301', '150', '273', '131', '169', '128', '174', '94', '211', '281', '148', '261', '225', '230', '217', '119', '27', '251', '117', '244', '125']


In [19]:
section_slices = ['C57BL6J-1.092', 'C57BL6J-3.009']

ob_author_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="subclass_transfer",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Fiber_tracts",
    section_slices=section_slices,
)

ob_leiden_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="cell_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Fiber_tracts",
    section_slices=section_slices,
)


for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_author_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/Fiber_tracts_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_leiden_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/Fiber_tracts_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 16,375
background cells: 151,512
selected 16 groups from subclass_transfer:
['NLL-po Pax7 Gaba', 'L6b CTX Glut', 'SPVC Ccdc172 Glut', 'OEC NN', 'PB Evx2 Glut', 'Astro-CB NN', 'NLL Gata3 Gly-Gaba', 'OB-STR-CTX Inh IMN', 'L6b/CT ENT Glut', 'Oligo NN', 'CBN Dmbx1 Gaba', 'SPVC Nmu Glut', 'STN-PSTN Pitx2 Glut', 'NLL-SOC Spp1 Glut', 'CT SUB Glut', 'PG-TRN-LRN Fat2 Glut']
target cells: 16,375
background cells: 151,512
selected 30 groups from cell_leiden:
['100', '291', '282', '103', '95', '134', '272', '293', '277', '84', '197', '189', '141', '104', '36', '256', '210', '269', '207', '251', '140', '180', '331', '53', '47', '44', '98', '270', '204', '316']


In [20]:
section_slices = ['C57BL6J-2.055', 'C57BL6J-1.117']

ob_author_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="subclass_transfer",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Midbrain",
    section_slices=section_slices,
)

ob_leiden_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="cell_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Midbrain",
    section_slices=section_slices,
)


for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_author_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/Midbrain_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_leiden_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/Midbrain_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 15,526
background cells: 47,791
selected 30 groups from subclass_transfer:
['IC Tfap2d Maf Glut', 'SCsg Pde5a Glut', 'SC Tnnt1 Gli3 Gaba', 'SCsg Gabrr2 Gaba', 'PAG Pou4f1 Ebf2 Glut', 'IC Six3 En2 Gaba', 'PAG-MRN Tfap2b Glut', 'PAG Pou4f3 Glut', 'SCs Pax7 Nfia Gaba', 'CUN Evx2 Lhx2 Glut', 'PAG Pou4f1 Bnc2 Glut', 'PAG Pou4f2 Mesi2 Glut', 'LDT Vsx2 Nkx6-1 Nfib Glut', 'PAG-SC Pou4f1 Zic1 Glut', 'SC Bnc2 Glut', 'SCiw Pitx2 Glut', 'PAG Pou4f2 Glut', 'PAG-PPN Pax5 Sox21 Gaba', 'PAG-RN Nkx2-2 Otx1 Gaba', 'IPN-LDT Vsx2 Nkx6-1 Glut', 'SCs Dmbx1 Gaba', 'SC-PAG Lef1 Emx2 Gaba', 'SCig Foxb1 Glut', 'SCm-PAG Cdh23 Gaba', 'PAG-SC Neurod2 Meis2 Glut', 'MRN-PPN-CUN Pax8 Gaba', 'SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut', 'MRN-VTN-PPN Pax5 Cdh23 Gaba', 'Astroependymal NN', 'PAG-MRN Pou3f1 Glut']
target cells: 15,526
background cells: 47,791
selected 15 groups from cell_leiden:
['289', '329', '59', '285', '73', '158', '274', '205', '43', '140', '41', '132', '16', '35', '3']


In [21]:
section_slices = ['C57BL6J-1.078', 'C57BL6J-3.012']

ob_author_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="subclass_transfer",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Thalamus",
    section_slices=section_slices,
)

ob_leiden_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="cell_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Thalamus",
    section_slices=section_slices,
)


for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'subclass_transfer', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = subclass_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_author_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/Thalamus_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'cell_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = cell_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = ob_leiden_groups,
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_cell/Thalamus_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 15,343
background cells: 135,566
selected 30 groups from subclass_transfer:
['VMH Fezf1 Glut', 'RE-Xi Nox4 Glut', 'VMH Nr5a1 Glut', 'CM-IAD-CL-PCN Sema5b Glut', 'ARH-PVp Tbx3 Glut', 'DMH-LHA Vgll2 Glut', 'PH-ant-LHA Otp Bsx Glut', 'ARH-PVp Tbx3 Gaba', 'TU-ARH Otp Six6 Gaba', 'TH Prkcd Grin2c Glut', 'PVHd-DMH Lhx6 Gaba', 'SBPV-PVa Six6 Satb2 Gaba', 'DMH Hmx2 Gaba', 'PVT-PT Ntrk1 Glut', 'PVHd-SBPV Six3 Prox1 Gaba', 'DMH-LHA Gsx1 Gaba', 'PVH-SO-PVa Otp Glut', 'LGv-SPFp-SPFm Nkx2-2 Tcf7l2 Gaba', 'Tanycyte NN', 'AHN-SBPV-PVHd Pdrm12 Gaba', 'RT-ZI Gnb3 Gaba', 'ZI Pax6 Gaba', 'SPA-SPFm-SPFp-POL-PIL-PoT Sp9 Glut', 'LH Pou4f1 Sox1 Glut', 'LGv-ZI Otx2 Gaba', 'PH-LHA Foxb1 Glut', 'MH Tac2 Glut', 'STN-PSTN Pitx2 Glut', 'Astro-NT NN', 'BST-MPN Six3 Nrgn Gaba']
target cells: 15,343
background cells: 135,566
selected 30 groups from cell_leiden:
['206', '320', '82', '297', '102', '186', '144', '90', '337', '257', '159', '205', '198', '16', '254', '121', '163', '35', '152', '209', '80', '

In [22]:
cell_leiden_palette

{'339': '#d67dcd',
 '121': '#58ad79',
 '2': '#c995d6',
 '221': '#d054ff',
 '132': '#ffd15a',
 '83': '#5fffd3',
 '57': '#65ffd4',
 '144': '#ffd66b',
 '140': '#d871ff',
 '99': '#d977ff',
 '12': '#ffdb7d',
 '41': '#82ffdd',
 '36': '#88ffde',
 '92': '#ffe08e',
 '210': '#d0bcd8',
 '123': '#ff8fd9',
 '66': '#eecae2',
 '215': '#3bffda',
 '251': '#a440ff',
 '97': '#edff46',
 '71': '#ff764b',
 '128': '#ff50bc',
 '261': '#5dff55',
 '95': '#5b91ff',
 '148': '#6095ff',
 '64': '#6cff65',
 '149': '#ff6ac6',
 '113': '#ff9270',
 '237': '#f2ff75',
 '27': '#c07aff',
 '107': '#7fffe7',
 '23': '#b0d4cd',
 '31': '#856d64',
 '278': '#a55133',
 '129': '#b59b91',
 '286': '#335b91',
 '151': '#929ba8',
 '296': '#ff8fa7',
 '168': '#d48fff',
 '336': '#ff6d51',
 '317': '#57ff71',
 '333': '#765cff',
 '322': '#7a62ff',
 '287': '#67ff7f',
 '316': '#ff836d',
 '310': '#ff8872',
 '255': '#77ff8d',
 '161': '#917dff',
 '306': '#9682ff',
 '252': '#88ff9a',
 '50': '#ff9f8d',
 '267': '#e4b6ae',
 '190': '#33adad',
 '220': '#3